# 1. Package Imports section

In [0]:
import re
import logging
import pyspark.sql.functions as F
from pyspark.sql import Window

# 2. Dataset Configs

In [0]:
# logging configuration
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s %(levelname)s %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S",
)
logger = logging.getLogger("Silver_Layer_Subcouncil_Arrears")

#silver tables config
ds_config = {
        "silver_table": "cpt_utility_catalog.silver.silver_service_requests_cleaned",
        "bronze_table": "cpt_utility_catalog.bronze.bronze_service_requests_raw",
        "changes": {
            "headers": {
                "column_mapping":{
                    "Sub_Council": "subcouncil",
                    "Ward": "ward",
                    "Suburb": "suburb",
                    "C3_Complaint_Type": "c3_complaint_type",
                    "Work_Center": "work_center",
                    "Notification": "notification",
                    "Notification_type": "notification_type",
                    "X_Y_Co_ordinate_1": "longitude_x",
                    "X_Y_Co_ordinate_2": "latitude_y",
                    "Created_On_Date": "created_on_date",
                    "Changed_on": "changed_on_date",
                    "Completed_Date": "completed_on_date",
                    "Notifications_Created": "notifications_created",
                }
            },
            "columns": {
                 "data_types":{
                    "amount": "decimal(13,2)",
                    "date": "date",
                    "ageing_bucket_days": "string",
                    "id": "string",
                    "subcouncil": "int",
                },
                "columns_to_drop": ["ObjectId"],
                "fill_na_value": None, 
            },
            "trim": True,
            "drop_columns": True,
            "drop_duplicates": True,
            "write_to_table": True,
            "rename_headers": True,
            "unpivot": True,
            "add_id": True,
            "cast_type": True,
            "col_cleanse": True,
        },
    }

df_new = spark.table(ds_config["bronze_table"])
hdr_config = ds_config["changes"]["headers"]
col_config = ds_config["changes"]["columns"]
changes = ds_config["changes"]

logger.info("\t- Silver layer subcouncil arrears table configuration loaded")


# 3. Dataset Cleaning

## 3.1 Drop Columns

In [0]:
if changes["drop_columns"] and isinstance(col_config["columns_to_drop"], list):
    logger.info("Dropping column(s)")
    # dropping columns that won't be needed acccording to config
    df_new = df_new.drop(*col_config["columns_to_drop"])
    logger.info("\t- Column(s) dropped")

## 3.2 Rename Headers

In [0]:
if changes["rename_headers"]:
    logger.info("Renaming headers")
    hdr_transformations = dict()
    # enrishing dataset with sensible names
    if isinstance(hdr_config["column_mapping"], dict):

        df_new = df_new.select(
        [F.col(col).alias(hdr_config["column_mapping"].get(col, col)) for col in df_new.columns]
    )

    logger.info("\t- Header(s) renamed")

## 3.3 Trim Whitespace

In [0]:
if changes["trim"]:
    logger.info("Trimming column(s)")
    # trimming column values whilst they're all casted string
    df_new = df_new.select([F.trim(F.col(col)).alias(col) for col in df_new.columns])
    logger.info("\t- Column(s) trimmed")

## 3.4 Column Cleanse

In [0]:
if changes["col_cleanse"]:
        logger.info("Cleaning column(s)")
        
        bad_placeholders = ["#", "N/A", "NA", "null", "NULL", "None", "", "-", "?","#_Not assigned", "Not assigned","*"]

        # Replace any occurrence of these placeholders with SQL NULL
        df_new = df_new.replace(bad_placeholders, None)

        logger.info("\t- Column(s) cleaned")

df_new.display()